# ECA&D Precipitation pipeline — Windows

Runs locally on Windows. No Colab, no Drive.

**Before running:** set `BASE_DIR` in the next cell to the folder holding
`ECA_blend_rr.zip`, then run cells in order.

Requirements:

```
pip install pandas pyarrow
```

`pyarrow` is only needed for the daily parquet export at the end.

In [1]:
import re
import zipfile
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------
# CONFIGURE THIS
#
# Point BASE_DIR at the folder containing ECA_blend_rr.zip.
# Use a raw string (r"...") so Windows backslashes are not read
# as escape characters.
# ---------------------------------------------------------------

BASE_DIR = Path(r"C:\projects\ccai-transformer-viability")

ZIP_PATH = BASE_DIR / "ECA_blend_rr.zip"
EXTRACT_DIR = BASE_DIR / "data\\rr"
OUTPUT_DIR = BASE_DIR / "processed"
DAILY_DIR = OUTPUT_DIR / "rr_daily"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Zip exists:", ZIP_PATH.exists())

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {ZIP_PATH}\n"
        f"Edit BASE_DIR above to point at the folder holding ECA_blend_rr.zip."
    )

Base directory: C:\projects\ccai-transformer-viability
Zip exists: False


FileNotFoundError: Could not find C:\projects\ccai-transformer-viability\ECA_blend_rr.zip
Edit BASE_DIR above to point at the folder holding ECA_blend_rr.zip.

In [2]:
# Extract the archive. Skipped if it has already been unpacked,
# so re-running this cell is cheap.

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

already = list(EXTRACT_DIR.glob("RR_STAID*.txt"))

if already:
    print(f"Already extracted: {len(already):,} files in {EXTRACT_DIR}")
else:
    print("Extracting (this takes a minute)...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)
    print("Extracted to:", EXTRACT_DIR)

Already extracted: 17,612 files in C:\projects\ccai-transformer-viability\data\rr


In [3]:
rr_files = sorted(
    EXTRACT_DIR.rglob("RR_STAID*.txt")
)

station_files = list(
    EXTRACT_DIR.rglob("stations.txt")
)

print("Precipitation station files:", len(rr_files))
print("stations.txt files found:", len(station_files))

if not station_files:
    raise FileNotFoundError("Could not find stations.txt")

if not rr_files:
    raise FileNotFoundError(
        "No RR_STAID files found. Check that the zip extracted correctly."
    )

stations_path = station_files[0]

print("Using stations file:", stations_path)
print("Example precipitation file:", rr_files[0])

Precipitation station files: 17612
stations.txt files found: 1
Using stations file: C:\projects\ccai-transformer-viability\data\rr\stations.txt
Example precipitation file: C:\projects\ccai-transformer-viability\data\rr\RR_STAID000001.txt


In [4]:
def make_header_re(var: str) -> re.Pattern:
    return re.compile(
        rf"^\s*STAID\s*,\s*SOUID\s*,\s*DATE\s*,\s*{var}\s*,\s*Q_{var}",
        re.IGNORECASE
    )


def parse_station_file(path: Path, var: str = "RR") -> pd.DataFrame:
    """
    Parse a single ECA&D blended station file.

    Generic over the variable code, so the same function works for
    RR (precipitation) and TG (temperature). The raw value column is
    named "<var>_raw" in lowercase, e.g. RR -> rr_raw.

    Encoding is pinned to latin-1 rather than left to the platform
    default. On Windows that default is cp1252, which raises on some
    station names.
    """
    value_col = f"{var.lower()}_raw"
    q_col = f"Q_{var}"
    header_re = make_header_re(var)

    # Find where the actual data table begins
    with open(path, "r", encoding="latin-1", errors="replace") as f:
        header_idx = next(
            (i for i, line in enumerate(f) if header_re.match(line)),
            None
        )

    if header_idx is None:
        return pd.DataFrame(
            columns=[
                "station_id",
                "source_id",
                "date",
                value_col,
                "quality"
            ]
        )

    # Read the actual data table
    df = pd.read_csv(
        path,
        skiprows=header_idx + 1,
        header=None,
        names=[
            "station_id",
            "source_id",
            "date",
            value_col,
            q_col
        ],
        skipinitialspace=True,
        encoding="latin-1"
    )

    # Convert data types
    df["station_id"] = pd.to_numeric(
        df["station_id"],
        errors="coerce"
    )

    df["source_id"] = pd.to_numeric(
        df["source_id"],
        errors="coerce"
    )

    df[value_col] = pd.to_numeric(
        df[value_col],
        errors="coerce"
    )

    df[q_col] = pd.to_numeric(
        df[q_col],
        errors="coerce"
    )

    df["date"] = pd.to_datetime(
        df["date"],
        format="%Y%m%d",
        errors="coerce"
    )

    df = df.rename(
        columns={q_col: "quality"}
    )

    return df.dropna(
        subset=["station_id", "date"]
    )

In [5]:
test_df = parse_station_file(
    rr_files[0],
    var="RR"
)

test_df.head()

,station_id,source_id,date,rr_raw,quality
0,1,37886,1860-01-01,0,0
1,1,37886,1860-01-02,0,0
2,1,37886,1860-01-03,136,0
3,1,37886,1860-01-04,49,0
4,1,37886,1860-01-05,0,0


In [6]:
def dms_to_decimal(value):
    if pd.isna(value):
        return float("nan")

    value = str(value).strip()

    sign = -1 if value.startswith("-") else 1

    value = value.lstrip("+-")

    degrees, minutes, seconds = map(
        float,
        value.split(":")
    )

    return sign * (
        degrees
        + minutes / 60
        + seconds / 3600
    )

In [7]:
def read_stations_file(path: Path):

    with open(path, "r", encoding="latin-1", errors="replace") as f:
        header_idx = next(
            (
                i for i, line in enumerate(f)
                if line.strip().startswith("STAID,STANAME")
            ),
            None
        )

    if header_idx is None:
        raise ValueError("Could not find stations.txt header")

    stations = pd.read_csv(
        path,
        skiprows=header_idx,
        skipinitialspace=True,
        encoding="latin-1"
    )

    stations.columns = stations.columns.str.strip()

    stations["STAID"] = pd.to_numeric(
        stations["STAID"],
        errors="coerce"
    )

    stations["latitude"] = stations["LAT"].apply(
        dms_to_decimal
    )

    stations["longitude"] = stations["LON"].apply(
        dms_to_decimal
    )

    return (
        stations[
            ["STAID", "latitude", "longitude"]
        ]
        .rename(
            columns={"STAID": "station_id"}
        )
        .dropna(subset=["station_id"])
        .drop_duplicates(subset=["station_id"])
    )

In [8]:
station_locations = read_stations_file(
    stations_path
)

station_locations.head()

,station_id,latitude,longitude
0,1,56.866667,14.800000
1,2,60.616667,15.616667
2,3,65.066667,17.166389
3,4,58.400000,15.533056
4,5,58.400000,15.533056


In [9]:
# Paper period: 2010-2021.
#
# Twelve years, matching the train/calibration/val/test split
# (2010-2018 / 2019 / 2020 / 2021). Post-2021 ECA&D years are often
# preliminary rather than fully blended, and are outside every split.
START_DATE = pd.Timestamp("2010-01-01")
END_DATE = pd.Timestamp("2021-12-31")

QUALITY_THRESHOLD = 0.90

# Precipitation-specific: a monthly TOTAL is only meaningful if most
# days in the month are present. A month with 12 valid days does not
# give a comparable total to a month with 31. Station-months below
# this coverage are dropped rather than rescaled, so no synthetic
# rainfall is introduced.
MIN_MONTH_COVERAGE = 0.90

# Wet-day threshold in mm (WMO convention: >= 1.0 mm)
WET_DAY_THRESHOLD_MM = 1.0

# Write daily observations alongside the monthly summary. The parse
# work is identical either way, so this costs nothing extra and saves
# a second full pass over ~10,000 files.
EXPORT_DAILY = True
DAILY_FLUSH_EVERY = 500  # stations per parquet part

In [13]:
import shutil

quality_results = []
monthly_results = []

daily_buffer = []
daily_part = 0

if EXPORT_DAILY:
    if DAILY_DIR.exists():
        shutil.rmtree(DAILY_DIR)
    DAILY_DIR.mkdir(parents=True, exist_ok=True)


def flush_daily():
    global daily_buffer, daily_part
    if not daily_buffer:
        return
    part = pd.concat(daily_buffer, ignore_index=True)
    part.to_parquet(
        DAILY_DIR / f"part_{daily_part:04d}.parquet",
        index=False,
        engine="fastparquet",
    )
    daily_part += 1
    daily_buffer = []


for i, file_path in enumerate(rr_files, start=1):

    try:
        df = parse_station_file(file_path, var="RR")

        if df.empty:
            continue

        period = df[
            (df["date"] >= START_DATE) &
            (df["date"] <= END_DATE)
        ].copy()

        if period.empty:
            continue

        station_id = int(period["station_id"].iloc[0])

        # ------------------------------------------------
        # Good observations
        # ------------------------------------------------

        good_mask = (
            (period["quality"] == 0) &
            (period["rr_raw"] != -9999) &
            (period["rr_raw"] >= 0) &
            period["rr_raw"].notna()
        )

        n_total = len(period)
        n_good = int(good_mask.sum())

        pct_good = n_good / n_total if n_total > 0 else None

        quality_results.append({
            "station_id": station_id,
            "file": file_path.name,
            "n_records": n_total,
            "n_good_quality": n_good,
            "pct_good_quality": pct_good,
        })

        if pct_good is None or pct_good <= QUALITY_THRESHOLD:
            continue

        valid = period.loc[good_mask].copy()

        # Convert RR to millimetres
        #   154 -> 15.4 mm
        #     0 ->  0.0 mm (dry day, a real value)
        valid["rr"] = valid["rr_raw"] / 10.0

        # ------------------------------------------------
        # Daily export
        # ------------------------------------------------

        if EXPORT_DAILY:
            daily_buffer.append(valid[["station_id", "source_id", "date", "rr"]])
            if len(daily_buffer) >= DAILY_FLUSH_EVERY:
                flush_daily()

        # ------------------------------------------------
        # Monthly aggregation
        # ------------------------------------------------

        # NOTE: deliberately NOT using .dt.to_period("M") here. Under
        # pyarrow, repeatedly creating pandas Period arrays inside a tight
        # loop that also calls to_parquet() eventually trips a known bug
        # (apache/arrow#41857): pyarrow's extension-type registry gets
        # corrupted on some later iteration ("A type extension with name
        # pandas.period already defined"), and every station after that
        # point fails identically for the rest of the run. This does the
        # same truncation via plain numpy and never touches that code path.
        valid["month"] = valid["date"].values.astype("datetime64[M]")

        monthly = (
            valid
            .groupby(["station_id", "month"], as_index=False)
            .agg(
                source_id=(
                    "source_id",
                    lambda x: (
                        x.mode().iloc[0]
                        if not x.mode().empty
                        else x.iloc[0]
                    )
                ),

                # Precipitation accumulates: monthly statistic is a SUM
                rr=("rr", "sum"),

                n_valid_days=("rr", "size"),

                n_wet_days=(
                    "rr",
                    lambda s: int(
                        (s >= WET_DAY_THRESHOLD_MM).sum()
                    )
                )
            )
        )

        monthly["days_in_month"] = monthly["month"].dt.days_in_month

        monthly["month_coverage"] = (
            monthly["n_valid_days"] / monthly["days_in_month"]
        )

        monthly = monthly[
            monthly["month_coverage"] >= MIN_MONTH_COVERAGE
        ]

        if monthly.empty:
            continue

        monthly_results.append(monthly)

        if i % 500 == 0:
            print(f"Processed {i:,}/{len(rr_files):,} station files")

    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")


if EXPORT_DAILY:
    flush_daily()
    print(f"\nDaily parts written: {daily_part} -> {DAILY_DIR}")

print("Done.")

Processed 500/17,612 station files
Processed 2,000/17,612 station files
Processed 2,500/17,612 station files
Processed 3,000/17,612 station files
Processed 3,500/17,612 station files
Processed 4,500/17,612 station files
Processed 7,000/17,612 station files
Processed 7,500/17,612 station files
Processed 8,000/17,612 station files
Processed 9,000/17,612 station files
Processed 9,500/17,612 station files
Processed 10,500/17,612 station files
Processed 11,500/17,612 station files
Processed 12,000/17,612 station files
Processed 13,000/17,612 station files
Processed 14,500/17,612 station files
Processed 15,000/17,612 station files
Processed 16,000/17,612 station files
Processed 17,000/17,612 station files

Daily parts written: 21 -> C:\projects\ccai-transformer-viability\processed\rr_daily
Done.


In [14]:
quality_df = (
    pd.DataFrame(quality_results)
    .sort_values("station_id")
    .reset_index(drop=True)
)

quality_df.head()

,station_id,file,n_records,n_good_quality,pct_good_quality
0,1,RR_STAID000001.txt,4383,3061,0.69838
1,2,RR_STAID000002.txt,4383,4383,1.00000
2,3,RR_STAID000003.txt,4383,0,0.00000
3,4,RR_STAID000004.txt,4383,4383,1.00000
4,5,RR_STAID000005.txt,4383,4383,1.00000


In [15]:
good_stations = quality_df[
    quality_df["pct_good_quality"] > QUALITY_THRESHOLD
].copy()

print("Stations evaluated:", len(quality_df))
print("Stations above 90%:", len(good_stations))
print(
    "Percentage retained:",
    round(len(good_stations) / len(quality_df) * 100, 2),
    "%"
)

Stations evaluated: 17509
Stations above 90%: 10142
Percentage retained: 57.92 %


In [16]:
if len(monthly_results) == 0:
    raise RuntimeError(
        "No stations passed the >90% quality threshold."
    )

monthly_data = pd.concat(monthly_results, ignore_index=True)

monthly_data.head()

,station_id,month,source_id,rr,n_valid_days,n_wet_days,days_in_month,month_coverage
0,2,2010-01-01,39422,20.1,31,7,31,1.0
1,2,2010-02-01,39422,42.2,28,12,28,1.0
2,2,2010-03-01,39422,34.0,31,6,31,1.0
3,2,2010-04-01,39422,10.1,30,5,30,1.0
4,2,2010-05-01,39422,104.8,31,9,31,1.0


In [17]:
monthly_data = monthly_data.merge(
    station_locations,
    on="station_id",
    how="left"
)

monthly_data = monthly_data.rename(columns={"month": "date"})

monthly_data.head()

,station_id,date,source_id,rr,n_valid_days,n_wet_days,days_in_month,month_coverage,latitude,longitude
0,2,2010-01-01,39422,20.1,31,7,31,1.0,60.616667,15.616667
1,2,2010-02-01,39422,42.2,28,12,28,1.0,60.616667,15.616667
2,2,2010-03-01,39422,34.0,31,6,31,1.0,60.616667,15.616667
3,2,2010-04-01,39422,10.1,30,5,30,1.0,60.616667,15.616667
4,2,2010-05-01,39422,104.8,31,9,31,1.0,60.616667,15.616667


In [18]:
monthly_data = monthly_data[
    [
        "station_id",
        "source_id",
        "date",
        "latitude",
        "longitude",
        "rr",
        "n_valid_days",
        "n_wet_days"
    ]
]

monthly_data = (
    monthly_data
    .sort_values(["date", "station_id"])
    .reset_index(drop=True)
)

monthly_data.head(20)

,station_id,source_id,date,latitude,longitude,rr,n_valid_days,n_wet_days
0,2,39422,2010-01-01,60.616667,15.616667,20.1,31,7
1,4,38500,2010-01-01,58.400000,15.533056,35.2,31,8
2,5,38500,2010-01-01,58.400000,15.533056,35.2,31,8
3,10,38884,2010-01-01,59.350000,18.050000,26.2,31,5
4,11,2147,2010-01-01,48.055000,14.130833,32.5,31,8
5,12,35,2010-01-01,47.077778,15.448889,34.0,31,7
6,13,37,2010-01-01,47.260000,11.384167,12.4,31,6
7,14,39,2010-01-01,47.789167,13.008056,28.7,31,9
8,15,41,2010-01-01,47.054167,12.957500,113.8,31,19
9,16,43,2010-01-01,48.248333,16.356389,61.7,31,12


In [19]:
print("Shape:", monthly_data.shape)
print("Number of stations:", monthly_data["station_id"].nunique())
print("Number of months:", monthly_data["date"].nunique())
print(
    "Date range:",
    monthly_data["date"].min(),
    "to",
    monthly_data["date"].max()
)

print("\nMissing values:")
print(monthly_data.isna().sum())

Shape: (1420766, 8)
Number of stations: 10142
Number of months: 144
Date range: 2010-01-01 00:00:00 to 2021-12-01 00:00:00

Missing values:
station_id      0
source_id       0
date            0
latitude        0
longitude       0
rr              0
n_valid_days    0
n_wet_days      0
dtype: int64


## Sanity check

In [20]:
print(monthly_data["rr"].describe())

print("\nNegative monthly totals (should be 0):",
      int((monthly_data["rr"] < 0).sum()))

print("Zero-total station-months:",
      int((monthly_data["rr"] == 0).sum()))

print("Max monthly total (mm):",
      round(monthly_data["rr"].max(), 1))

print("Mean valid days per month:",
      round(monthly_data["n_valid_days"].mean(), 2))

by_month = (
    monthly_data
    .groupby(monthly_data["date"].dt.month)["rr"]
    .mean()
    .round(1)
)

print("\nMean total by calendar month (mm):")
print(by_month)

count    1.420766e+06
mean     6.689374e+01
std      5.553320e+01
min      0.000000e+00
25%      3.080000e+01
50%      5.360000e+01
75%      8.740000e+01
max      1.172800e+03
Name: rr, dtype: float64

Negative monthly totals (should be 0): 0
Zero-total station-months: 10285
Max monthly total (mm): 1172.8
Mean valid days per month: 30.42

Mean total by calendar month (mm):
date
1     68.1
2     53.7
3     50.8
4     43.7
5     71.0
6     73.1
7     76.9
8     80.3
9     67.9
10    74.2
11    70.4
12    72.7
Name: rr, dtype: float64


In [21]:
duplicates = monthly_data.duplicated(
    subset=["station_id", "date"]
).sum()

print("Duplicate station-month rows:", duplicates)

Duplicate station-month rows: 0


In [22]:
output_path = OUTPUT_DIR / "monthly_precipitation_2010_2021.csv"

monthly_data.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: C:\projects\ccai-transformer-viability\processed\monthly_precipitation_2010_2021.csv


## Daily output

The monthly CSV above is a station-level QC artifact. The daily parquet
below is what feeds the model — monthly totals cannot support the
extreme-event split, since those are within-month events.

In [24]:
daily_parts = sorted(DAILY_DIR.glob("part_*.parquet"))

print("Parquet parts:", len(daily_parts))

if not daily_parts:
    raise FileNotFoundError(
        "No daily parts found. Was EXPORT_DAILY set to True before "
        "running the processing loop?"
    )

daily_data = pd.concat(
    [pd.read_parquet(p, engine="fastparquet") for p in daily_parts],
    ignore_index=True
)

daily_data = daily_data.merge(
    station_locations,
    on="station_id",
    how="left"
)

daily_data = (
    daily_data
    .sort_values(["date", "station_id"])
    .reset_index(drop=True)
)

print("Daily shape:", daily_data.shape)
print("Stations:", daily_data["station_id"].nunique())
print("Days:", daily_data["date"].nunique())
print(
    "Date range:",
    daily_data["date"].min(),
    "to",
    daily_data["date"].max()
)

daily_data.head()

Parquet parts: 21
Daily shape: (43439104, 6)
Stations: 10142
Days: 4383
Date range: 2010-01-01 00:00:00 to 2021-12-31 00:00:00


,station_id,source_id,date,rr,latitude,longitude
0,2,39422,2010-01-01,1.1,60.616667,15.616667
1,4,38500,2010-01-01,1.4,58.400000,15.533056
2,5,38500,2010-01-01,1.4,58.400000,15.533056
3,10,38884,2010-01-01,0.7,59.350000,18.050000
4,11,2147,2010-01-01,5.1,48.055000,14.130833


In [ ]:
# Shrink dtypes before writing. Saves roughly half the file size and
# most of the memory when this is loaded for training.

daily_data["station_id"] = daily_data["station_id"].astype("int32")
daily_data["source_id"] = daily_data["source_id"].astype("int32")
daily_data["rr"] = daily_data["rr"].astype("float32")
daily_data["latitude"] = daily_data["latitude"].astype("float32")
daily_data["longitude"] = daily_data["longitude"].astype("float32")

daily_path = OUTPUT_DIR / "daily_precipitation_2010_2021.parquet"

daily_data.to_parquet(daily_path, index=False, engine="fastparquet")

print("Saved to:", daily_path)
print("Size:", round(daily_path.stat().st_size / 1e6, 1), "MB")

Saved to: C:\projects\ccai-transformer-viability\processed\daily_precipitation_2010_2021.parquet
Size: 688.0 MB


In [27]:
daily_path = OUTPUT_DIR / "daily_precipitation_2010_2021.csv"

daily_data.to_csv(daily_path, index=False)

print("Saved to:", daily_path)
print("Size:", round(daily_path.stat().st_size / 1e6, 1), "MB")

Saved to: C:\projects\ccai-transformer-viability\processed\daily_precipitation_2010_2021.csv
Size: 1994.3 MB


In [28]:
%reset

## Station coverage over the paper period

Completeness measured against calendar days in 2010–2021, not against
records present in the file. These are different quantities, and the
two halves of the project need to agree on which one the 90% threshold
refers to.

In [ ]:
DAYS_2010_2021 = 4383  # 12 years, 3 leap days

coverage = (
    daily_data
    .groupby("station_id")["date"]
    .nunique()
    / DAYS_2010_2021
)

print(coverage.describe())
print("\nStations below 0.90 coverage:", int((coverage < 0.90).sum()))
print("Stations below 0.80 coverage:", int((coverage < 0.80).sum()))

valid_stations = coverage[coverage >= 0.90].index.to_series(name="station_id")

valid_path = OUTPUT_DIR / "valid_precipitation_stations.csv"
valid_stations.to_csv(valid_path, index=False)

print("\nStations retained:", len(valid_stations))
print("Saved to:", valid_path)

## Next step

Run the other notebook, then intersect the two station lists. Both
variables must use the same stations — otherwise any difference between
temperature and precipitation results is partly a difference in which
stations were measured, and the comparison is uncontrolled.

```python
import pandas as pd

rr = set(pd.read_csv(OUTPUT_DIR / "valid_precipitation_stations.csv")["station_id"])
tg = set(pd.read_csv(OUTPUT_DIR / "valid_temperature_stations.csv")["station_id"])

both = sorted(rr & tg)
print("RR:", len(rr), " TG:", len(tg), " intersection:", len(both))

pd.Series(both, name="station_id").to_csv(
    OUTPUT_DIR / "valid_stations_shared.csv", index=False
)
```